In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/upi_transactions_2024.csv")
df = df.rename(columns={"transaction id": "transaction_id", "amount (INR)": "amount_inr"})
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df['timestamp'] = pd.to_datetime(df['timestamp'])

df.to_csv("../data/processed/cleaned_transactions.csv", index=False)
print(df.shape)

(250000, 17)


In [2]:
import sqlite3
df = pd.read_csv("../data/processed/cleaned_transactions.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# --- dim_date: generate full calendar year, not just observed dates ---
dates = pd.date_range("2024-01-01", "2024-12-31", freq="D")
dim_date = pd.DataFrame({
    "date_key": dates.strftime("%Y%m%d").astype(int),
    "full_date": dates.strftime("%Y-%m-%d"),
    "year": dates.year,
    "month": dates.month,
    "month_name": dates.strftime("%B"),
    "day_name": dates.strftime("%A"),
    "is_weekend": (dates.dayofweek >= 5).astype(int),
    "quarter": dates.quarter,
    "week_of_year": dates.isocalendar().week.values
})

# --- small dimensions built from unique values ---
def build_dim(df, col, key_name):
    dim = pd.DataFrame({col: sorted(df[col].unique())})
    dim[key_name] = range(1, len(dim) + 1)
    return dim

dim_geography = build_dim(df, "sender_state", "sender_state_key")
dim_merchant_category = build_dim(df, "merchant_category", "merchant_category_key")
dim_transaction_type = build_dim(df, "transaction_type", "transaction_type_key")
dim_device = build_dim(df, "device_type", "device_type_key")
dim_network = build_dim(df, "network_type", "network_type_key")
dim_age_group = build_dim(df, "sender_age_group", "age_group_key")

# --- fact table: merge in surrogate keys (ONE merge chain, using original names) ---
fact = df.copy()
fact["date_key"] = fact["timestamp"].dt.strftime("%Y%m%d").astype(int)
fact = fact.merge(dim_geography, on="sender_state") \
           .merge(dim_merchant_category, on="merchant_category") \
           .merge(dim_transaction_type, on="transaction_type") \
           .merge(dim_device, on="device_type") \
           .merge(dim_network, on="network_type") \
           .merge(dim_age_group, on="sender_age_group")

fact["transaction_key"] = range(1, len(fact) + 1)
fact_transactions = fact[[
    "transaction_key", "transaction_id", "date_key", "hour_of_day",
    "sender_state_key", "merchant_category_key", "transaction_type_key",
    "device_type_key", "network_type_key", "age_group_key",
    "amount_inr", "transaction_status", "fraud_flag"
]]

# sanity check before loading — row count must match source
assert len(fact_transactions) == len(df), "Row count mismatch after merges — a join is dropping/duplicating rows"
print(f"Fact table rows: {len(fact_transactions)}")

# NOW rename dim_age_group's column, only after it's done its merge job
dim_age_group = dim_age_group.rename(columns={"sender_age_group": "age_group"})

Fact table rows: 250000


In [3]:
import os

# clean slate — avoid duplicate rows if the DB already has a partial load from earlier attempts
if os.path.exists("../warehouse/payments_warehouse.db"):
    os.remove("../warehouse/payments_warehouse.db")

os.makedirs("../warehouse", exist_ok=True)
os.makedirs("../sql", exist_ok=True)

schema_sql = """
CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL,
    year INTEGER,
    month INTEGER,
    month_name TEXT,
    day_name TEXT,
    is_weekend INTEGER,
    quarter INTEGER,
    week_of_year INTEGER
);

CREATE TABLE dim_geography (
    sender_state_key INTEGER PRIMARY KEY,
    sender_state TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_merchant_category (
    merchant_category_key INTEGER PRIMARY KEY,
    merchant_category TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_transaction_type (
    transaction_type_key INTEGER PRIMARY KEY,
    transaction_type TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_device (
    device_type_key INTEGER PRIMARY KEY,
    device_type TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_network (
    network_type_key INTEGER PRIMARY KEY,
    network_type TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_age_group (
    age_group_key INTEGER PRIMARY KEY,
    age_group TEXT NOT NULL UNIQUE
);

CREATE TABLE fact_transactions (
    transaction_key INTEGER PRIMARY KEY,
    transaction_id TEXT NOT NULL UNIQUE,
    date_key INTEGER NOT NULL,
    hour_of_day INTEGER,
    sender_state_key INTEGER,
    merchant_category_key INTEGER,
    transaction_type_key INTEGER,
    device_type_key INTEGER,
    network_type_key INTEGER,
    age_group_key INTEGER,
    amount_inr NUMERIC NOT NULL,
    transaction_status TEXT NOT NULL,
    fraud_flag INTEGER NOT NULL,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (sender_state_key) REFERENCES dim_geography(sender_state_key),
    FOREIGN KEY (merchant_category_key) REFERENCES dim_merchant_category(merchant_category_key),
    FOREIGN KEY (transaction_type_key) REFERENCES dim_transaction_type(transaction_type_key),
    FOREIGN KEY (device_type_key) REFERENCES dim_device(device_type_key),
    FOREIGN KEY (network_type_key) REFERENCES dim_network(network_type_key),
    FOREIGN KEY (age_group_key) REFERENCES dim_age_group(age_group_key)
);

CREATE INDEX idx_fact_date ON fact_transactions(date_key);
CREATE INDEX idx_fact_state ON fact_transactions(sender_state_key);
CREATE INDEX idx_fact_status ON fact_transactions(transaction_status);
CREATE INDEX idx_fact_fraud ON fact_transactions(fraud_flag);
"""

with open("../sql/schema.sql", "w") as f:
    f.write(schema_sql)

conn = sqlite3.connect("../warehouse/payments_warehouse.db")
conn.executescript(schema_sql)

dim_date.to_sql("dim_date", conn, if_exists="append", index=False)
dim_geography.to_sql("dim_geography", conn, if_exists="append", index=False)
dim_merchant_category.to_sql("dim_merchant_category", conn, if_exists="append", index=False)
dim_transaction_type.to_sql("dim_transaction_type", conn, if_exists="append", index=False)
dim_device.to_sql("dim_device", conn, if_exists="append", index=False)
dim_network.to_sql("dim_network", conn, if_exists="append", index=False)
dim_age_group.to_sql("dim_age_group", conn, if_exists="append", index=False)
fact_transactions.to_sql("fact_transactions", conn, if_exists="append", index=False)

conn.commit()

# verification query
check = pd.read_sql("""
    SELECT g.sender_state, COUNT(*) as txn_count, SUM(f.amount_inr) as tpv
    FROM fact_transactions f
    JOIN dim_geography g ON f.sender_state_key = g.sender_state_key
    GROUP BY g.sender_state
    ORDER BY tpv DESC
""", conn)
print(check)
print("Total rows across states:", check['txn_count'].sum())

conn.close()

     sender_state  txn_count       tpv
0     Maharashtra      37427  49043948
1   Uttar Pradesh      30125  40035717
2       Karnataka      29756  38451158
3      Tamil Nadu      25367  33343518
4           Delhi      24870  32689865
5       Telangana      22435  29750930
6       Rajasthan      19981  26730470
7         Gujarat      20061  25988190
8  Andhra Pradesh      20006  25952619
9     West Bengal      19972  25952594
Total rows across states: 250000
